In [1]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import OneHotEncoder, StandardScaler

import tensorflow as tf
from tensorflow.keras import layers

In [2]:
data1 = pd.DataFrame({
    "edad": np.random.randint(21, 65, 1000),
    "sexo": np.random.choice(["M", "F"], 1000),
    "ingreso": np.random.lognormal(mean=10, sigma=0.5, size=1000),
    "profesion": np.random.choice(
        ["ingeniero", "doctor", "abogado", "profesor"], 1000
    )
})

data2 = pd.DataFrame({
    "edad": np.random.randint(18, 65, 100),
    "sexo": np.random.choice(["M", "F"], 100),
    "ingreso": 0,
    "profesion": np.random.choice(
        ["desempleado"], 100
    )
})

data3 = pd.DataFrame({
    "edad": np.random.randint(3, 21, 200),
    "sexo": np.random.choice(["M", "F"], 200),
    "ingreso": 0,
    "profesion": np.random.choice(
        ["estudiante"], 200
    )
})

data = pd.concat([data1, data2, data3])
data.reset_index(drop=True, inplace=True)
data

,edad,sexo,ingreso,profesion
0,42,M,15421.429725,profesor
1,56,M,18546.750318,abogado
2,50,M,35708.154585,ingeniero
3,40,F,48755.820434,ingeniero
4,25,F,33540.670571,ingeniero
...,...,...,...,...
1295,18,F,0.000000,estudiante
1296,13,M,0.000000,estudiante
1297,15,F,0.000000,estudiante
1298,5,M,0.000000,estudiante


In [3]:
categorical_cols = ["sexo", "profesion"]
continuous_cols = ["edad", "ingreso"]

# Log transform ingreso
data["ingreso"] = np.log(data["ingreso"] + 1)

ohe = OneHotEncoder(sparse_output=False)
scaler = StandardScaler()

X_cat = ohe.fit_transform(data[categorical_cols])
X_cont = scaler.fit_transform(data[continuous_cols])

X = np.concatenate([X_cont, X_cat], axis=1)
X = X.astype("float32")

data_dim = X.shape[1]

In [9]:
data[categorical_cols]

,sexo,profesion
0,M,profesor
1,M,abogado
2,M,ingeniero
3,F,ingeniero
4,F,ingeniero
...,...,...
1295,F,estudiante
1296,M,estudiante
1297,F,estudiante
1298,M,estudiante


In [12]:
ohe.categories_

[array(['F', 'M'], dtype=object),
 array(['abogado', 'desempleado', 'doctor', 'estudiante', 'ingeniero',
        'profesor'], dtype=object)]

In [4]:
X.shape

(1300, 10)

In [5]:
cat_dims = [len(ohe.categories_[i]) for i in range(len(categorical_cols))]
cond_dim = sum(cat_dims)

def sample_condition(batch_size):
    cond = np.zeros((batch_size, cond_dim))
    for i, dim in enumerate(cat_dims):
        idx = np.random.randint(0, dim, batch_size)
        start = sum(cat_dims[:i])
        cond[np.arange(batch_size), start + idx] = 1
    return tf.convert_to_tensor(cond, dtype=tf.float32)


In [6]:
print(cat_dims, cond_dim)

[2, 6] 8


In [7]:
sample_condition(5)

2026-02-12 18:49:20.656316: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: CUDA_ERROR_NO_DEVICE: no CUDA-capable device is detected


<tf.Tensor: shape=(5, 8), dtype=float32, numpy=
array([[1., 0., 0., 0., 0., 0., 0., 1.],
       [0., 1., 0., 0., 0., 1., 0., 0.],
       [1., 0., 0., 0., 1., 0., 0., 0.],
       [1., 0., 0., 0., 0., 0., 1., 0.],
       [1., 0., 0., 0., 0., 1., 0., 0.]], dtype=float32)>

In [8]:
latent_dim = 128

def build_generator():
    z = layers.Input(shape=(latent_dim,))
    cond = layers.Input(shape=(cond_dim,))
    x = layers.Concatenate()([z, cond])

    x = layers.Dense(256, activation="relu")(x)
    x = layers.Dense(256, activation="relu")(x)
    out = layers.Dense(data_dim)(x)

    return tf.keras.Model([z, cond], out)


In [9]:
def build_discriminator():
    x_in = layers.Input(shape=(data_dim,))
    cond = layers.Input(shape=(cond_dim,))
    x = layers.Concatenate()([x_in, cond])

    x = layers.Dense(256, activation="relu")(x)
    x = layers.Dense(256, activation="relu")(x)
    out = layers.Dense(1)(x)

    return tf.keras.Model([x_in, cond], out)


In [10]:
def gradient_penalty(discriminator, real, fake, cond):
    alpha = tf.random.uniform([real.shape[0], 1], 0., 1.)
    interpolated = alpha * real + (1 - alpha) * fake

    with tf.GradientTape() as tape:
        tape.watch(interpolated)
        pred = discriminator([interpolated, cond], training=True)

    grads = tape.gradient(pred, interpolated)
    norm = tf.sqrt(tf.reduce_sum(tf.square(grads), axis=1))
    return tf.reduce_mean((norm - 1.0) ** 2)


In [ ]:
#max(f)

max(f) = min(-f)


In [11]:
generator = build_generator()
discriminator = build_discriminator()

g_opt = tf.keras.optimizers.Adam(1e-5)
d_opt = tf.keras.optimizers.Adam(1e-5)

batch_size = 256
epochs = 300
lambda_gp = 10
critic_steps = 5

dataset = tf.data.Dataset.from_tensor_slices(X).shuffle(2000).batch(batch_size)

for epoch in range(epochs):
    for real_batch in dataset:
        for _ in range(critic_steps):
            z = tf.random.normal([real_batch.shape[0], latent_dim])
            cond = sample_condition(real_batch.shape[0])

            with tf.GradientTape() as tape:
                fake = generator([z, cond], training=True)
                d_real = discriminator([real_batch, cond], training=True)
                d_fake = discriminator([fake, cond], training=True)

                gp = gradient_penalty(discriminator, real_batch, fake, cond)
                d_loss = tf.reduce_mean(d_fake) - tf.reduce_mean(d_real) + lambda_gp * gp

            grads = tape.gradient(d_loss, discriminator.trainable_variables)
            d_opt.apply_gradients(zip(grads, discriminator.trainable_variables))

        z = tf.random.normal([real_batch.shape[0], latent_dim])
        cond = sample_condition(real_batch.shape[0])

        with tf.GradientTape() as tape:
            fake = generator([z, cond], training=True)
            g_loss = -tf.reduce_mean(discriminator([fake, cond], training=True))

        grads = tape.gradient(g_loss, generator.trainable_variables)
        g_opt.apply_gradients(zip(grads, generator.trainable_variables))

    if epoch % 50 == 0:
        print(f"Epoch {epoch} | D: {d_loss.numpy():.3f} | G: {g_loss.numpy():.3f}")


2026-02-09 15:51:40.241049: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


Epoch 0 | D: 2.380 | G: 0.045


2026-02-09 15:51:42.030752: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
2026-02-09 15:51:45.026674: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
2026-02-09 15:51:50.747844: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
2026-02-09 15:52:02.126410: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
2026-02-09 15:52:27.757947: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


Epoch 50 | D: -0.327 | G: -0.712


2026-02-09 15:53:22.979075: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


Epoch 100 | D: -0.866 | G: -0.190


2026-02-09 15:55:13.302199: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


Epoch 150 | D: -0.908 | G: 0.724
Epoch 200 | D: -0.793 | G: 2.001
Epoch 250 | D: -1.004 | G: 3.146


2026-02-09 15:58:31.059555: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


In [12]:
z = tf.random.normal([1000, latent_dim])
cond = sample_condition(1000)

synthetic = generator([z, cond], training=False).numpy()

X_cont_syn = scaler.inverse_transform(synthetic[:, :len(continuous_cols)])
X_cat_syn = ohe.inverse_transform(synthetic[:, len(continuous_cols):])

synthetic_df = pd.DataFrame(
    np.column_stack([X_cont_syn, X_cat_syn]),
    columns=continuous_cols + categorical_cols
)

synthetic_df['ingreso'] = np.exp(synthetic_df['ingreso'].astype(float)) 

print(synthetic_df.head(50))


         edad       ingreso sexo    profesion
0   61.023239  18142.891102    F    ingeniero
1   45.879436  26044.589940    F    ingeniero
2    43.57494  33364.092287    M     profesor
3   49.753239  64773.411032    F     profesor
4   31.571564  67988.268968    M      abogado
5   25.424524   5491.978928    F       doctor
6   38.503674   5290.361066    F     profesor
7   37.332924  47503.763775    F       doctor
8   49.166969  22618.611373    M      abogado
9   42.472519   1156.880610    F     profesor
10  18.867281   6181.485232    F     profesor
11  22.248909   2026.983620    F    ingeniero
12  34.029182     86.170297    M     profesor
13  44.710087  22052.381519    M    ingeniero
14  40.642982  20628.879459    M    ingeniero
15  11.201211    241.733386    M   estudiante
16  45.607002  33093.371342    F      abogado
17  39.503185   8514.795249    F    ingeniero
18  45.334824  49594.915653    F    ingeniero
19  14.511119      1.244216    M   estudiante
20  33.706749   7738.095119    M  

In [13]:
print(synthetic_df[synthetic_df["profesion"] == "desempleado"])

          edad      ingreso sexo    profesion
43   50.965771   246.097727    F  desempleado
138  33.283497     0.130963    F  desempleado
233  40.195435    46.141762    M  desempleado
256  -4.468087     0.208371    M  desempleado
344  42.256222   454.649749    M  desempleado
365  38.952747   378.778152    M  desempleado
494  35.176212    10.242417    M  desempleado
495  42.850182   120.928665    M  desempleado
578  24.392273   445.684916    M  desempleado
614  50.143829    24.784428    F  desempleado
628  24.280977     7.336166    F  desempleado
712  42.326195    21.683062    F  desempleado
714  30.009398   426.991101    M  desempleado
881  29.789629   143.943095    F  desempleado
889  49.130718  3843.907877    F  desempleado
